# Data Wrangling Dataset 1: Katalog 120 Menu Tujuan Semata

In [1]:
import re
import pandas as pd
import numpy as np

In [3]:
df_raw = pd.read_csv("raw_katalog_menu_tujuan_semata.csv")
print(f"Dimensi Data Mentah: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")

Dimensi Data Mentah: 132 baris x 11 kolom


## Assessing Data (Identifikasi Masalah Kualitas Data Mentah)
 Memeriksa duplikasi `Kode_Menu`, nilai kosong, ketidakkonsistenan penulisan huruf/spasi pada `Nama Menu Produk` & `Kategori`, format mata uang pada `Harga_Jual` & `Estimasi_Modal_HPP`, serta format boolean campuran pada `Status_Menu_Baru` dan `Menu_Terlaris`.

In [4]:
print("=== Jumlah Missing Values (NaN) pada Data Mentah ===")
print(df_raw.isna().sum())

kode_norm = df_raw['Kode_Menu'].astype(str).str.strip().str.upper()
print(f"\nJumlah Kode_Menu Unik: {kode_norm.nunique()} menu")
print(f"Jumlah Baris Duplikat Kode_Menu: {kode_norm.duplicated().sum()} baris")

=== Jumlah Missing Values (NaN) pada Data Mentah ===
Kode_Menu                0
Nama Menu Produk         0
Kategori                 0
Sub_Kategori             9
Harga_Jual               0
Estimasi_Modal_HPP      18
Komposisi / Bahan       12
Karakter_Rasa            9
Status_Menu_Baru        21
Menu_Terlaris            9
Catatan_Khusus_Owner     0
dtype: int64

Jumlah Kode_Menu Unik: 120 menu
Jumlah Baris Duplikat Kode_Menu: 12 baris


## Cleaning Data (Pembersihan, Standarisasi, & Imputasi Kontekstual)
Tahapan pembersihan dilakukan secara deterministik:
1. **Deduplikasi `Kode_Menu`**: Menstandarisasi ke huruf kapital (`MN001`–`MN120`) dan menghapus **12 baris duplikat** sehingga tersisa tepat **120 menu unik**.
2. **Standarisasi `Nama_Menu` & `Kategori`**: Menghapus spasi berlebih dan merapikan kapitalisasi agar 100% cocok saat di-*merge* dengan `Menu_Disebut_Ulasan` di Dataset 3.
3. **Konversi `Harga_Jual` & Imputasi `Estimasi_Modal_HPP`**: Mengubah format `"Rp 21.000.-"`, `"22k"`, `"20rb"` menjadi integer murni, serta mengimputasi 16 baris `Estimasi_Modal_HPP` yang kosong dengan rumus konsisten **`0.42 * Harga_Jual` (42%)**.
4. **Imputasi `Sub_Kategori`, `Komposisi_Bahan`, & `Karakter_Rasa`**: Mengisi nilai kosong sesuai pemetaan kategori dan deskripsi menu.
5. **Standarisasi Boolean `Status_Menu_Baru` & `Menu_Terlaris`**: Menyeragamkan nilai `True/1/Ya/YES/Baru/Best Seller` menjadi boolean `True`/`False` dan mengimputasi baris kosong dari `Catatan_Khusus_Owner`.

In [5]:
df = df_raw.copy()

# Standarisasi Kode_Menu & Hapus 12 Baris Duplikat
df['Kode_Menu'] = df['Kode_Menu'].astype(str).str.strip().str.upper()
df = df.drop_duplicates(subset=['Kode_Menu'], keep='first').sort_values('Kode_Menu').reset_index(drop=True)

In [6]:
# Standarisasi Nama_Menu (Title Case untuk baris yang huruf kecil semua)
df['Nama_Menu'] = df['Nama Menu Produk'].apply(lambda x: str(x).strip().title() if str(x).strip().islower() else str(x).strip())

In [ ]:
# Standarisasi 8 Kategori Utama
kategori_map = {
    'signature drink (cokelat kental)': 'Signature Drink (Cokelat Kental)',
    'coffee & espresso based': 'Coffee & Espresso Based',
    'non-coffee, milk & matcha': 'Non-Coffee, Milk & Matcha',
    'kids & family friendly menu': 'Kids & Family Friendly Menu',
    'tea, mocktail & refresher': 'Tea, Mocktail & Refresher',
    'main course (eatery makanan berat)': 'Main Course (Eatery Makanan Berat)',
    'snack & sharing tongkrongan': 'Snack & Sharing Tongkrongan',
    'dessert, toast & sweet bites': 'Dessert, Toast & Sweet Bites'
}
df['Kategori'] = df['Kategori'].astype(str).str.strip().str.lower().map(kategori_map)

In [7]:
# Imputasi Sub_Kategori berdasarkan Kategori (7 baris NaN)
sub_kategori_map = {
    'Signature Drink (Cokelat Kental)': 'Minuman Dingin',
    'Coffee & Espresso Based': 'Kopi',
    'Non-Coffee, Milk & Matcha': 'Minuman Non-Kopi',
    'Kids & Family Friendly Menu': 'Menu Anak & Keluarga',
    'Tea, Mocktail & Refresher': 'Minuman Segar',
    'Main Course (Eatery Makanan Berat)': 'Makanan Berat',
    'Snack & Sharing Tongkrongan': 'Camilan Gurih',
    'Dessert, Toast & Sweet Bites': 'Makanan Penutup'
}
df['Sub_Kategori'] = df['Sub_Kategori'].fillna(df['Kategori'].map(sub_kategori_map))


In [8]:
# Konversi Harga_Jual & Imputasi Estimasi_Modal_HPP (42% dari Harga_Jual)
def parse_rupiah(val):
    if pd.isna(val):
        return np.nan
    s = str(val).lower().strip()
    if 'k' in s or 'rb' in s:
        return int(re.sub(r'[^0-9]', '', s)) * 1000
    digits = re.sub(r'[^0-9]', '', s)
    return int(digits) if digits else np.nan

df['Harga_Jual'] = df['Harga_Jual'].apply(parse_rupiah).astype(int)
df['Estimasi_Modal_HPP'] = df['Estimasi_Modal_HPP'].apply(parse_rupiah)
df['Estimasi_Modal_HPP'] = df['Estimasi_Modal_HPP'].fillna((df['Harga_Jual'] * 0.42).round()).astype(int)

In [9]:
# Imputasi Komposisi_Bahan (11 baris NaN) & Karakter_Rasa (7 baris N/A)
imputasi_komposisi = {
    'MN016': 'Cokelat bubuk dark premium, susu full cream hangat, kental manis',
    'MN022': 'Espresso shot, saus butterscotch gurih manis, sirup esens rum halal, susu segar',
    'MN043': 'Bubuk ubi ungu (taro) lembut, susu full cream segar, simple syrup',
    'MN045': 'Bubuk arang aktif (activated charcoal) food grade, sirup vanila, susu segar',
    'MN056': 'Bubuk red velvet lembut, remahan biskuit Oreo, susu segar blend dingin',
    'MN066': 'Teh melati seduh segar, sirup leci harum, buah leci utuh, es batu porsi jumbo',
    'MN075': 'Seduhan bunga telang biru (butterfly pea) alami, perasan jeruk lemon segar, madu/simple syrup',
    'MN077': 'Nasi putih hangat, irisan daging sapi (sliced beef) saus teriyaki Jepang, wijen, telur',
    'MN081': 'Nasi putih hangat, ayam goreng tepung crispy, saus telur asin (salted egg) creamy, daun kari',
    'MN104': '4 potong sayap ayam goreng crispy, saus gochujang pedas manis ala Korea, taburan wijen',
    'MN106': 'Irisan bawang bombay goreng tepung crispy (onion rings), saus tartar mayo lemon'
}
df['Komposisi_Bahan'] = df['Komposisi / Bahan']
for kode, komp in imputasi_komposisi.items():
    df.loc[df['Kode_Menu'] == kode, 'Komposisi_Bahan'] = df.loc[df['Kode_Menu'] == kode, 'Komposisi_Bahan'].fillna(komp)

imputasi_rasa = {
    'MN001': 'Non-Kopi, Manis Pekat, Cokelat Kental, Creamy',
    'MN005': 'Non-Kopi, Manis Harum Pisang, Creamy',
    'MN013': 'Non-Kopi, Manis Lembut Ubi Ungu, Creamy',
    'MN088': 'Pasta, Pedas Gurih Bawang Putih, Savoury Tuna',
    'MN090': 'Western Main Course, Gurih Renyah, Segar Tartar Lemon',
    'MN116': 'Roti Bakar, Manis Harum Pandan Srikaya, Gurih Butter',
    'MN118': 'Dessert Renyah, Manis Kayu Manis, Cocolan Cokelat Hangat'
}
df['Karakter_Rasa'] = df['Karakter_Rasa'].replace('N/A', np.nan)
for kode, rasa in imputasi_rasa.items():
    df.loc[df['Kode_Menu'] == kode, 'Karakter_Rasa'] = df.loc[df['Kode_Menu'] == kode, 'Karakter_Rasa'].fillna(rasa)

In [10]:
# Standarisasi Boolean Status_Menu_Baru & Menu_Terlaris
def clean_status_baru(row):
    val = str(row['Status_Menu_Baru']).strip().lower()
    catatan = str(row['Catatan_Khusus_Owner']).strip().lower()
    if val in ['true', '1', '1.0', 'ya', 'yes', 'baru']:
        return True
    if val in ['false', '0', '0.0', 'tidak', 'no']:
        return False
    return 'menu baru' in catatan

def clean_menu_terlaris(row):
    val = str(row['Menu_Terlaris']).strip().lower()
    catatan = str(row['Catatan_Khusus_Owner']).strip().lower()
    if val in ['true', '1', '1.0', 'ya', 'yes', 'best seller']:
        return True
    if val in ['false', '0', '0.0', 'tidak', 'no']:
        return False
    return any(k in catatan for k in ['best seller', 'paling laris', 'favorit'])

df['Status_Menu_Baru'] = df.apply(clean_status_baru, axis=1)
df['Menu_Terlaris'] = df.apply(clean_menu_terlaris, axis=1)
df['Catatan_Khusus_Owner'] = df['Catatan_Khusus_Owner'].astype(str).str.strip()

In [11]:
# QA VALIDATION & EXPORT DATASET BERSIH
kolom_final = [
    'Kode_Menu', 'Nama_Menu', 'Kategori', 'Sub_Kategori',
    'Harga_Jual', 'Estimasi_Modal_HPP', 'Komposisi_Bahan',
    'Karakter_Rasa', 'Status_Menu_Baru', 'Menu_Terlaris', 'Catatan_Khusus_Owner'
]
df_clean = df[kolom_final].copy()
df_clean.to_csv("katalog_menu_tujuan_semata.csv", index=False)

print(f"\nDimensi Data Bersih: {df_clean.shape[0]} baris x {df_clean.shape[1]} kolom")
print("Total Missing Values:", df_clean.isna().sum().sum())


Dimensi Data Bersih: 120 baris x 11 kolom
Total Missing Values: 1


# Data Wrangling Dataset 3: Ulasan & Chat Pelanggan Tujuan Semata

In [12]:
df_raw = pd.read_csv("raw_ulasan_dan_chat_tujuan_semata.csv")

print(f"Dimensi awal (baris, kolom): {df_raw.shape}\n")
print("5 Baris Pertama Data Mentah:")
print(df_raw.head(5)[['ID_Log', 'Tanggal_Masuk', 'Sumber_Ulasan', 'Skor_Rating_Mentah', 'Label_Sentimen_Awal', 'Label_Intent_Mentah']])

Dimensi awal (baris, kolom): (2581, 10)

5 Baris Pertama Data Mentah:
         ID_Log Tanggal_Masuk               Sumber_Ulasan Skor_Rating_Mentah  \
0  RAW-LOG-0001    2026-10-02  DM Instagram @tujuansemata                  5   
1  RAW-LOG-0001    2026-10-02  DM Instagram @tujuansemata                  5   
2  RAW-LOG-0002    2026-07-26          Google Review Maps                  4   
3  RAW-LOG-0003    2026-06-17         Chat WhatsApp Kedai                  4   
4  RAW-LOG-0004    2026-05-10  DM Instagram @tujuansemata                  5   

  Label_Sentimen_Awal         Label_Intent_Mentah  
0             positif  tanya_produk_dan_menu_baru  
1             positif  tanya_produk_dan_menu_baru  
2         Positif (+)              pemesanan_menu  
3             positif  rekomendasi_dan_membership  
4             positif                         NaN  


##  Assessing Data (Penilaian Kualitas & Struktur Data Mentah)
 Memeriksa tipe data, *missing values* (`NaN`), 81 baris duplikat pada `ID_Log`, inkonsistensi format `Skor_Rating_Mentah` (`"4 Bintang"`), `Label_Sentimen_Awal` (`"Positif (+)"`), nilai `"Belum Dicek"` pada `Catatan_Komplain_Mentah`, serta *noise* HTML `<br>`, URL, huruf kapital berlebih, dan kata *slang* pada teks.

In [13]:
print("\nJumlah Missing Values (NaN) per Kolom:")
print(df_raw.isnull().sum())


Jumlah Missing Values (NaN) per Kolom:
ID_Log                           0
Ref_Nota_Transaksi               0
Tanggal_Masuk                    0
Sumber_Ulasan                    0
Skor_Rating_Mentah              88
Teks_Ulasan_Mentah              90
Label_Sentimen_Awal            210
Catatan_Komplain_Mentah          0
Pesan_Chat_Pelanggan_Mentah      0
Label_Intent_Mentah            118
dtype: int64


In [14]:
print(f"\nDuplikasi ID_Log: {df_raw['ID_Log'].duplicated().sum()} baris")
for col in ['Skor_Rating_Mentah', 'Label_Sentimen_Awal', 'Catatan_Komplain_Mentah', 'Label_Intent_Mentah']:
    print(f"\n--- Distribusi Nilai Mentah: {col} ---")
    print(df_raw[col].value_counts(dropna=False))



Duplikasi ID_Log: 81 baris

--- Distribusi Nilai Mentah: Skor_Rating_Mentah ---
Skor_Rating_Mentah
5            1130
4             645
5 Bintang     318
4 Bintang     182
NaN            88
2              79
3              71
1              29
3 Bintang      19
2 Bintang      17
1 Bintang       3
Name: count, dtype: int64

--- Distribusi Nilai Mentah: Label_Sentimen_Awal ---
Label_Sentimen_Awal
positif        1501
Positif (+)     312
netral          296
NaN             210
negatif         145
Netral (0)       76
Negatif (-)      41
Name: count, dtype: int64

--- Distribusi Nilai Mentah: Catatan_Komplain_Mentah ---
Catatan_Komplain_Mentah
Tidak Ada Keluhan                    1788
Kapasitas Tempat / Variasi Menu       335
Belum Dicek                           248
Double Order / Pencatatan Manual       78
Salah Dengar / Salah Order Manual      71
Waktu Tunggu Saat Ramai                40
Sikap Pelayanan Saat Kelelahan         21
Name: count, dtype: int64

--- Distribusi Nilai Mentah: Labe

## Cleaning Data (Pembersihan, Standarisasi, Imputasi, & Feature Engineering)
 1. **Deduplikasi `ID_Log`**: Menghapus 81 baris duplikat sehingga tersisa tepat **2.500 baris unik**.
 2. **Standarisasi & Imputasi `Skor_Rating`, `Label_Sentimen`, `Catatan_Komplain`, & `Label_Intent`**: Mengubah string `"4 Bintang"` menjadi integer `4`, menyeragamkan sentimen ke `positif`/`netral`/`negatif`, serta mengisi nilai kosong dan status `"Belum Dicek"` berdasarkan pola kalimat ulasan/chat.
 3. **Feature Engineering (`Estimasi_Waktu_Tunggu_Menit` & `Menu_Disebut_Ulasan`)**: Mengekstrak angka durasi tunggu (menit) dan nama menu spesifik dari teks ulasan.
 4. **Pembersihan Noise Teks & Normalisasi NLP (`Teks_Ulasan`, `Teks_Ulasan_NLP`, `Pesan_Chat`, `Pesan_Chat_NLP`)**: Menghapus tag `<br>`, URL, *hashtag*, tanda seru berlebih, serta menormalisasi kata *slang* (`gw`/`aku` $\rightarrow$ `saya`, `bgt` $\rightarrow$ `banget`, `gak`/`nggak` $\rightarrow$ `tidak`).

In [15]:
# Menghapus 81 Baris Duplikat
df_clean = df_raw.drop_duplicates(subset=['ID_Log'], keep='first').copy().reset_index(drop=True)
df_clean['Ref_Nota_Transaksi'] = df_clean['Ref_Nota_Transaksi'].astype(str).str.strip()
df_clean['Tanggal_Masuk'] = pd.to_datetime(df_clean['Tanggal_Masuk'], format='%Y-%m-%d', errors='coerce')

In [16]:
# Standarisasi Format Skor Rating & Label Sentimen
def parse_rating(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'(\d+)', str(val))
    return float(m.group(1)) if m else np.nan

df_clean['Skor_Rating'] = df_clean['Skor_Rating_Mentah'].apply(parse_rating)

sent_map = {
    'positif': 'positif', 'Positif (+)': 'positif',
    'netral': 'netral', 'Netral (0)': 'netral',
    'negatif': 'negatif', 'Negatif (-)': 'negatif'
}
df_clean['Label_Sentimen'] = df_clean['Label_Sentimen_Awal'].map(sent_map)

In [17]:
# Imputasi Missing Values pada Label_Sentimen
def impute_sentiment(row):
    if pd.notna(row['Label_Sentimen']):
        return row['Label_Sentimen']
    t = str(row['Teks_Ulasan_Mentah']).lower() if pd.notna(row['Teks_Ulasan_Mentah']) else ''
    k = str(row['Catatan_Komplain_Mentah'])
    r = row['Skor_Rating']
    if any(x in t for x in ['double order', 'salah dengar', 'kurang senyum', 'antrean pembuatan']):
        return 'negatif'
    if k in ['Double Order / Pencatatan Manual', 'Salah Dengar / Salah Order Manual', 'Sikap Pelayanan Saat Kelelahan']:
        return 'negatif'
    if any(x in t for x in ['cukup oke buat teman nugas', 'meja bagian dalam lumayan cepat penuh', 'cuma tadi pas jam']):
        return 'netral'
    if k == 'Kapasitas Tempat / Variasi Menu':
        return 'netral'
    if pd.notna(r):
        if r >= 4:
            return 'positif'
        elif r == 3:
            return 'netral' if k == 'Waktu Tunggu Saat Ramai' else 'negatif'
        else:
            return 'negatif'
    return 'positif'

df_clean['Label_Sentimen'] = df_clean.apply(impute_sentiment, axis=1)

In [18]:
# Membersihkan Status 'Belum Dicek' pada Catatan_Komplain
def clean_komplain(row):
    k = str(row['Catatan_Komplain_Mentah'])
    if k != 'Belum Dicek':
        return k
    t = str(row['Teks_Ulasan_Mentah']).lower() if pd.notna(row['Teks_Ulasan_Mentah']) else ''
    s = row['Label_Sentimen']
    if 'double order' in t:
        return 'Double Order / Pencatatan Manual'
    elif 'salah dengar' in t:
        return 'Salah Dengar / Salah Order Manual'
    elif 'kurang senyum' in t or 'capek urus kedai' in t:
        return 'Sikap Pelayanan Saat Kelelahan'
    elif 'antrean pembuatan' in t:
        return 'Waktu Tunggu Saat Ramai'
    elif any(x in t for x in ['cukup oke buat teman nugas', 'meja bagian dalam lumayan cepat penuh', 'cuma tadi pas jam']):
        return 'Kapasitas Tempat / Variasi Menu'
    if s == 'positif':
        return 'Tidak Ada Keluhan'
    elif s == 'netral':
        return 'Kapasitas Tempat / Variasi Menu'
    else:
        return 'Double Order / Pencatatan Manual'

df_clean['Catatan_Komplain'] = df_clean.apply(clean_komplain, axis=1)

In [19]:
#  Imputasi Missing Values pada Skor_Rating
def impute_rating(row):
    if pd.notna(row['Skor_Rating']):
        return int(row['Skor_Rating'])
    s = row['Label_Sentimen']
    return 5 if s == 'positif' else (4 if s == 'netral' else 2)

df_clean['Skor_Rating'] = df_clean.apply(impute_rating, axis=1)

In [20]:
# Imputasi Missing Values pada Label_Intent
def impute_intent(row):
    if pd.notna(row['Label_Intent_Mentah']) and str(row['Label_Intent_Mentah']).strip() != '':
        return str(row['Label_Intent_Mentah']).strip()
    c = str(row['Pesan_Chat_Pelanggan_Mentah']).lower()
    if any(w in c for w in ['python', 'presiden', 'bitcoin', 'diferensial', 'linux', 'kontrak hukum', 'bengkel', 'romawi', 'gravitasi', 'esai akademik']):
        return 'out_of_context'
    elif any(w in c for w in ['salah, saya pesan', 'salah, gw pesan', 'double order', 'kasih rating bintang', 'antreannya agak lama', 'agak kemanisan', 'sudah diingatkan dengan jujur', 'kurang senyum', 'kurang saus sambalnya', 'nggak salah dengar pesanan lagi', 'kirim ulasan member']):
        return 'keluhan_dan_ulasan'
    elif any(w in c for w in ['mau pesan', 'mau order', 'tolong catat pesanan', 'pesan ', 'bantu input pesanan', 'mau beli', 'order via chatbot', 'tolong bikinin']):
        return 'pemesanan_menu'
    elif any(w in c for w in ['poin membership', 'rekomendasikan', 'rekomendasi', 'daftar akun membership', 'poin member', 'poin saya', 'poin gw', 'cocok buat ibu-ibu']):
        return 'rekomendasi_dan_membership'
    else:
        return 'tanya_produk_dan_menu_baru'

df_clean['Label_Intent'] = df_clean.apply(impute_intent, axis=1)

In [21]:
# Feature Engineering: Estimasi_Waktu_Tunggu_Menit & Menu_Disebut_Ulasan
def extract_wait_time(text):
    if pd.isna(text):
        return 0
    m = re.search(r'(\d+)\s*menit', str(text), flags=re.IGNORECASE)
    return int(m.group(1)) if m else 0

df_clean['Estimasi_Waktu_Tunggu_Menit'] = df_clean['Teks_Ulasan_Mentah'].apply(extract_wait_time)

canonical_menus = {}
patterns_menu = [
    r'komposisi\s+(.+?)\s+sebelum kami pesan',
    r'poin dari beli\s+(.+?)\s+bisa dikumpulin',
    r'dan menu\s+(.+?)-nya juara',
    r'Rasa\s+(.+?)\s+cukup oke buat teman nugas',
    r'Rasa\s+(.+?)\s+enak',
    r'Rekomendasi\s+(.+?)\s+dari chatbot',
    r'Sekarang pesan\s+(.+?)\s+lewat chatbot',
    r'Tadi pesan 1\s+(.+?)\s+tapi tercatat 2 kali',
    r'Menu\s+(.+?)-nya enak dengan harga',
    r'catat manual, pesan\s+(.+?)\s+malah keluar yang lain',
    r'antrean pembuatan\s+(.+?)\s+lumayan lama',
    r'tadi pas ditanya soal\s+(.+?)\s+kelihatan lelah',
]

for raw_t in df_clean['Teks_Ulasan_Mentah'].dropna():
    if not str(raw_t).strip().isupper():
        t_clean = re.sub(r'<br\s*/?>|https?://\S+|#\S+', ' ', str(raw_t), flags=re.IGNORECASE)
        t_clean = re.sub(r'\s+', ' ', t_clean).strip()
        for pat in patterns_menu:
            m = re.search(pat, t_clean, flags=re.IGNORECASE)
            if m:
                c_name = m.group(1).strip()
                canonical_menus[c_name.lower()] = c_name
                break

def extract_menu(text):
    if pd.isna(text) or str(text).strip() == "":
        return "Tidak Disebutkan"
    t = re.sub(r'<br\s*/?>|https?://\S+|#\S+', ' ', str(text), flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip()
    for pat in patterns_menu:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            menu_key = m.group(1).strip().lower()
            return canonical_menus.get(menu_key, m.group(1).strip().title())
    return "Tidak Disebutkan (Umum/Tempat)"

df_clean['Menu_Disebut_Ulasan'] = df_clean['Teks_Ulasan_Mentah'].apply(extract_menu)


In [22]:
# Imputasi & Normalisasi Teks Ulasan dan Pesan Chat
df_clean['Flag_Imputasi_Ulasan'] = df_clean['Teks_Ulasan_Mentah'].isna().astype(int)

def fill_missing_review_text(row):
    if pd.notna(row['Teks_Ulasan_Mentah']) and str(row['Teks_Ulasan_Mentah']).strip() != "":
        return str(row['Teks_Ulasan_Mentah'])
    k = row['Catatan_Komplain']
    s = row['Label_Sentimen']
    if s == 'positif':
        return "Pelayanan ramah dan menu Tujuan Semata sesuai ekspektasi pelanggan."
    elif k == 'Kapasitas Tempat / Variasi Menu':
        return "Tempatnya nyaman buat nongkrong sore, semoga kapasitas meja dan variasi menu di website makin ditambah."
    elif k == 'Waktu Tunggu Saat Ramai':
        return "Menu enak, namun waktu tunggu saat jam ramai agak lama karena dikerjakan sendiri."
    elif k == 'Double Order / Pencatatan Manual':
        return "Pesanan sempat tercatat dua kali (double order) saat antrean padat sehingga waktu tunggu lebih lama."
    elif k == 'Salah Dengar / Salah Order Manual':
        return "Saat kedai ramai pesanan sempat salah dengar karena pencatatan manual, namun sudah dibuatkan ulang."
    elif k == 'Sikap Pelayanan Saat Kelelahan':
        return "Pemilik tampak kelelahan saat mengurus kedai sendirian di jam sibuk sehingga kurang senyum."
    return "Ulasan pelanggan tercatat melalui evaluasi berkala."

SLANG_MAP = {
    r'\bgw\b': 'saya', r'\baku\b': 'saya', r'\bbgt\b': 'banget',
    r'\bkaaaak\b': 'kak', r'\bkakak\b': 'kak', r'\bmin\b': 'admin',
    r'\brame pol\b': 'ramai sekali', r'\brame\b': 'ramai',
    r'\bnggak\b': 'tidak', r'\bgak\b': 'tidak', r'\bsoalnya\b': 'karena',
    r'\bdihandle\b': 'ditangani', r'\bowner-nya\b': 'pemiliknya',
    r'\bowner\b': 'pemilik', r'\bbikinin\b': 'buatkan',
    r'\bgimana\b': 'bagaimana', r'\baja\b': 'saja',
}

def clean_text_readable(text):
    t = str(text)
    t = re.sub(r'<br\s*/?>|https?://\S+|#\S+', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'(\d{2})_(\d{2})', r'\1:\2', t)
    t = re.sub(r'\bkaaaak\b', 'kak', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(kak),\s*kak\b', r'\1', t, flags=re.IGNORECASE)
    t = re.sub(r'!+', '!', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t.capitalize() if t.isupper() else t

def clean_text_nlp(text):
    t = str(text).lower()
    t = re.sub(r'<br\s*/?>|https?://\S+|#\S+', ' ', t)
    t = re.sub(r'(\d{2})_(\d{2})', r'\1:\2', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    for pattern, repl in SLANG_MAP.items():
        t = re.sub(pattern, repl, t)
    t = re.sub(r'\b(kak)\s+kak\b', r'\1', t)
    return re.sub(r'\s+', ' ', t).strip()

df_clean['Teks_Ulasan'] = df_clean.apply(fill_missing_review_text, axis=1).apply(clean_text_readable)
df_clean['Teks_Ulasan_NLP'] = df_clean['Teks_Ulasan'].apply(clean_text_nlp)
df_clean['Pesan_Chat'] = df_clean['Pesan_Chat_Pelanggan_Mentah'].apply(clean_text_readable)
df_clean['Pesan_Chat_NLP'] = df_clean['Pesan_Chat'].apply(clean_text_nlp)

In [23]:
# QA VALIDATION & SIMPAN KE CSV BERSIH (15 KOLOM FINAL)

final_columns = [
    'ID_Log', 'Ref_Nota_Transaksi', 'Tanggal_Masuk', 'Sumber_Ulasan',
    'Skor_Rating', 'Label_Sentimen', 'Catatan_Komplain', 'Menu_Disebut_Ulasan',
    'Estimasi_Waktu_Tunggu_Menit', 'Flag_Imputasi_Ulasan', 'Teks_Ulasan',
    'Teks_Ulasan_NLP', 'Pesan_Chat', 'Pesan_Chat_NLP', 'Label_Intent'
]

df_wrangled = df_clean[final_columns].copy()
df_wrangled['Tanggal_Masuk'] = df_wrangled['Tanggal_Masuk'].dt.strftime('%Y-%m-%d')
df_wrangled.to_csv("ulasan_dan_chat_tujuan_semata.csv", index=False)

print("\n=== HASIL AKHIR VERIFIKASI DATA WRANGLING DATASET 3 ===")
print(f"Dimensi Dataset Bersih: {df_wrangled.shape}")
print("Total Missing Values:", df_wrangled.isnull().sum().sum())
print("Total Duplikasi ID_Log:", df_wrangled['ID_Log'].duplicated().sum())


=== HASIL AKHIR VERIFIKASI DATA WRANGLING DATASET 3 ===
Dimensi Dataset Bersih: (2500, 15)
Total Missing Values: 0
Total Duplikasi ID_Log: 0
